In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

# ── 1. 加载数据，重建 index ──
ratings = pd.read_csv("../data/ml-1m/ratings.dat", sep="::",
    names=["user_id","movie_id","rating","timestamp"], engine="python")
movies = pd.read_csv("../data/ml-1m/movies.dat", sep="::",
    names=["movie_id","title","genres"], engine="python", encoding="latin-1")

all_users  = sorted(ratings["user_id"].unique())
all_movies = sorted(ratings["movie_id"].unique())
user2idx  = {u: i for i, u in enumerate(all_users)}
movie2idx = {m: i for i, m in enumerate(all_movies)}
idx2movie = {i: m for m, i in movie2idx.items()}
movie_info = movies.set_index("movie_id")[["title","genres"]]

# ── 2. 定义 NCF（与训练时一致：含 Dropout，输出层名为 out）──
class NCF(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, mlp_dims=[64,32,16]):
        super().__init__()
        self.mf_user_emb   = nn.Embedding(n_users,  emb_dim)
        self.mf_movie_emb  = nn.Embedding(n_movies, emb_dim)
        self.mlp_user_emb  = nn.Embedding(n_users,  emb_dim)
        self.mlp_movie_emb = nn.Embedding(n_movies, emb_dim)
        layers = []
        in_dim = emb_dim * 2
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout()]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(emb_dim + mlp_dims[-1], 1)

    def forward(self, user_idx, movie_idx):
        mf = self.mf_user_emb(user_idx) * self.mf_movie_emb(movie_idx)
        mlp_in = torch.cat([self.mlp_user_emb(user_idx),
                            self.mlp_movie_emb(movie_idx)], dim=-1)
        mlp_out = self.mlp(mlp_in)
        out = self.out(torch.cat([mf, mlp_out], dim=-1))
        return torch.sigmoid(out).squeeze(-1)

# ── 3. 加载权重 ──
ckpt = torch.load("../data/ncf_clean.pth", map_location="cpu")
hp   = ckpt["hparams"]
ncf  = NCF(hp["n_users"], hp["n_movies"], hp["emb_dim"], hp["mlp_dims"])
ncf.load_state_dict(ckpt["model_state"])
ncf.eval()
print("NCF 加载成功，参数量:", sum(p.numel() for p in ncf.parameters()))

# ── 4. NCF 推荐函数 ──
def ncf_recommend(user_id, top_k=10):
    uidx     = user2idx[user_id]
    all_midx = torch.arange(len(movie2idx))
    uidx_t   = torch.full_like(all_midx, uidx)
    with torch.no_grad():
        scores = ncf(uidx_t, all_midx).numpy()
    seen     = set(ratings[ratings["user_id"] == user_id]["movie_id"].tolist())
    seen_idx = [movie2idx[m] for m in seen if m in movie2idx]
    scores[seen_idx] = -1.0
    top_idx  = np.argsort(scores)[::-1][:top_k]
    results  = []
    for idx in top_idx:
        mid = idx2movie[idx]
        results.append({
            "movie_id": mid,
            "title":    movie_info.loc[mid, "title"],
            "genres":   movie_info.loc[mid, "genres"],
            "score":    float(scores[idx])
        })
    return results

# ── 5. 验证 ──
for r in ncf_recommend(1, top_k=10):
    print(f"  {r['title']:<50} {r['genres']:<35} score={r['score']:.4f}")

NCF 加载成功，参数量: 630561
  American Beauty (1999)                             Comedy|Drama                        score=0.9501
  Matrix, The (1999)                                 Action|Sci-Fi|Thriller              score=0.9130
  L.A. Confidential (1997)                           Crime|Film-Noir|Mystery|Thriller    score=0.9113
  Silence of the Lambs, The (1991)                   Drama|Thriller                      score=0.9065
  Terminator 2: Judgment Day (1991)                  Action|Sci-Fi|Thriller              score=0.9000
  Men in Black (1997)                                Action|Adventure|Comedy|Sci-Fi      score=0.8999
  Jurassic Park (1993)                               Action|Adventure|Sci-Fi             score=0.8988
  Star Wars: Episode V - The Empire Strikes Back (1980) Action|Adventure|Drama|Sci-Fi|War   score=0.8984
  Being John Malkovich (1999)                        Comedy                              score=0.8966
  Pulp Fiction (1994)                                Crime

In [5]:
import sys
import os
sys.path.insert(0, "../src")

from recommender import Recommender

rec = Recommender(
    data_dir="../data",
    model_path="../data/two_tower_v2_best.pth",
    onnx_path="../data/user_tower.onnx"
)

recs = rec.recommend(user_id=1, top_k=10)
print("Two-Tower 推荐给用户 1：")
for r in recs["recommendations"]:
    print(f"  {r['title']:<50} score={r['score']:.4f}")

user tower: ONNX Runtime (../data/user_tower.onnx)
Recommender ready — 6040 users, 3706 items in Faiss index
Two-Tower 推荐给用户 1：
  Shakespeare in Love (1998)                         score=0.4878
  Galaxy Quest (1999)                                score=0.4319
  American Beauty (1999)                             score=0.4247
  Babe (1995)                                        score=0.4077
  Austin Powers: The Spy Who Shagged Me (1999)       score=0.3954
  Edward Scissorhands (1990)                         score=0.3869
  Forrest Gump (1994)                                score=0.3839
  Groundhog Day (1993)                               score=0.3781
  Men in Black (1997)                                score=0.3604
  Clueless (1995)                                    score=0.3604


In [6]:
week12_notes = """
**Week 12**：Streamlit UI + 收尾 ✅
- NCF 和 Two-Tower 双模型 A/B 对比 UI
- `st.cache_resource` 缓存模型，只在启动时加载一次
- Two-Tower 返回格式：`{"user_id": ..., "recommendations": [...]}`，genres 在 app 层补查
- 启动：`streamlit run src/app.py`

踩坑：
1. **NCF 定义与权重不匹配导致 kernel restart**：state_dict 里层名是 `out`（不是 `output_layer`），
   MLP 含 Dropout（下标跳3），定义必须和训练时一致。
   定位方法：terminal 里 `python -c` 复现，看到明确 RuntimeError，而不是 notebook 静默重启。
2. **Recommender 接口是 data_dir，不是分散的路径参数**：`Recommender(data_dir=..., model_path=..., onnx_path=...)`
3. **recommend 返回嵌套结构**：结果在 `resp["recommendations"]` 里，不是直接的列表

结果：
- NCF：动作/惊悚为主，sigmoid score 0.90+
- Two-Tower：喜剧/剧情为主，余弦相似度 score 0.39–0.49
- 重叠 1/6——内容特征显著改变了推荐方向
"""
print(week12_notes)


**Week 12**：Streamlit UI + 收尾 ✅
- NCF 和 Two-Tower 双模型 A/B 对比 UI
- `st.cache_resource` 缓存模型，只在启动时加载一次
- Two-Tower 返回格式：`{"user_id": ..., "recommendations": [...]}`，genres 在 app 层补查
- 启动：`streamlit run src/app.py`

踩坑：
1. **NCF 定义与权重不匹配导致 kernel restart**：state_dict 里层名是 `out`（不是 `output_layer`），
   MLP 含 Dropout（下标跳3），定义必须和训练时一致。
   定位方法：terminal 里 `python -c` 复现，看到明确 RuntimeError，而不是 notebook 静默重启。
2. **Recommender 接口是 data_dir，不是分散的路径参数**：`Recommender(data_dir=..., model_path=..., onnx_path=...)`
3. **recommend 返回嵌套结构**：结果在 `resp["recommendations"]` 里，不是直接的列表

结果：
- NCF：动作/惊悚为主，sigmoid score 0.90+
- Two-Tower：喜剧/剧情为主，余弦相似度 score 0.39–0.49
- 重叠 1/6——内容特征显著改变了推荐方向

